In [ ]:
from google.colab import userdata
for name in ["ANTHROPIC_API_KEY","OPENAI_API_KEY","GEMINI_API_KEY"]:
    try:
        v = userdata.get(name)
        print(f"{name}: {'OK, loaded' if v else 'EMPTY'} (length {len(v) if v else 0})")
    except Exception as e:
        print(f"{name}: NOT FOUND — {e}")

Tu peux insérer les remarques ici:


-

In [ ]:
# ============================================================
#  SINGLE-MODEL SCREENING PASS  (Colab)  — run once per model
#
#  Each model screens ALL records independently, writing ONLY its own
#  columns ({model}_verdict / _label / _conf / _notes) into a shared
#  results file.
#
#  RECOMMENDED ORDER:
#    1. Set MODEL_TO_RUN = "gpt"    -> run (fastest)
#    2. Set MODEL_TO_RUN = "claude" -> run in a second session
#    3. Set MODEL_TO_RUN = "gemini" -> run in a third session
#    4. Run 14b_consensus.py to combine all three
#
#  Each pass is safe to interrupt and resume — checkpoints are saved
#  every CHECKPOINT_EVERY records.
# ============================================================

# ── 0. COLAB SETUP ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(["pip", "install", "-q",
                "anthropic", "openai", "google-genai",
                "pandas", "openpyxl", "pyarrow"], check=True)

import os, re, json, time
import pandas as pd
from google.colab import userdata

# ── 1. CONFIGURATION ────────────────────────────────────────

# >>> CHANGE THIS for each pass <<<
MODEL_TO_RUN = "gpt"        # "gpt" | "claude" | "gemini"

# Model identifiers
CLAUDE_MODEL = "claude-sonnet-4-6"
GPT_MODEL    = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"

# Run behaviour
CHECKPOINT_EVERY = 500      # save progress every N records
MAX_RECORDS      = None # 1000     # set e.g. 50 for a quick test; None = all
RATE_LIMIT_SLEEP = 1.0      # seconds between API calls (~200/min)
                            # increase to 1.0 if you hit 429 errors

# Paths — edit FOLDER to match your Drive location
FOLDER  = "/content/drive/MyDrive/COGNOSX/TICKs_and_climate_change"
OUT_DIR = os.path.join(FOLDER, "preprocessing_data")
IN_PATH = os.path.join(OUT_DIR, "master_canonical_screen_ready.xlsx")  # input
RESULTS = os.path.join(OUT_DIR, f"screening_results_{MODEL_TO_RUN}.xlsx")              # shared output (all models)
CKPT    = os.path.join(OUT_DIR, f"screen_ckpt_{MODEL_TO_RUN}.parquet") # per-model resume file

# API keys — loaded from Colab Secrets only for the active model
ANTHROPIC_KEY = userdata.get("ANTHROPIC_API_KEY") if MODEL_TO_RUN == "claude" else ""
OPENAI_KEY    = userdata.get("OPENAI_API_KEY")    if MODEL_TO_RUN == "gpt"    else ""
GEMINI_KEY    = userdata.get("GEMINI_API_KEY")    if MODEL_TO_RUN == "gemini" else ""


# ── 2. SCREENING CRITERIA ────────────────────────────────────

CRITERIA = """You are an expert systematic reviewer in vector ecology and climate science.

Your task is to classify academic studies based on whether they analyze the relationship
between climate/environmental factors and ticks or tick-borne diseases.

─────────────────────────────────────────────
CLASSIFICATION LABELS
─────────────────────────────────────────────

1. CLIMATE_TICK_STUDY  ✅ INCLUDE
   The study uses climate, weather, seasonal, or environmental variables to explain, model,
   predict, or describe tick biology or tick-borne disease dynamics.

   Qualifying climate/environmental variables include:
   - Temperature, humidity, rainfall, precipitation, drought
   - Seasonality, phenology, seasonal abundance, overwintering
   - Climate change, global warming, greenhouse effect
   - Bioclimatic variables, habitat suitability, ecological niche
   - Vegetation index (NDVI), land use, altitude/elevation
   - Any meteorological or environmental factor linked to tick outcomes

   Qualifying tick outcomes include:
   - Tick distribution, range expansion, geographic spread, tick species, tick taxonomy
   - Tick abundance, density, population dynamics
   - Tick activity, questing behavior, phenology
   - Tick-borne pathogen prevalence or transmission risk driven by climate
   - Suitable habitat prediction for tick species

   NOTE: Seasonality IS a climate variable. A study analyzing tick abundance
   or pathogen detection by season satisfies this criterion even without
   explicit "climate change" framing.

2. TICK_STUDY_NO_CLIMATE  ❌ EXCLUDE
   The study concerns ticks or tick-borne diseases but does NOT use climate
   or environmental variables as explanatory or analytical factors.

   Examples:
   - Tick species surveillance or prevalence surveys with no climate analysis
   - Pathogen detection studies (PCR, serology) without climate variables
   - Clinical case series or treatment studies
   - Tick control or acaricide trials
   - Tick genomics, phylogenetics, or molecular taxonomy without climate link
   - Reviews of tick-borne pathogens without climate framing

3. CLIMATE_NO_TICK  ❌ EXCLUDE
   The study concerns climate change or environmental variables but does NOT
   involve ticks or tick-borne disease vectors.

4. NOT_RELEVANT  ❌ EXCLUDE
   The study is unrelated to both ticks and climate analysis.
   Examples: general infectious disease epidemiology, non-tick vector studies,
   plant or agricultural studies, purely clinical or pharmacological work.

─────────────────────────────────────────────
DECISION RULES
─────────────────────────────────────────────

RULE 1 — Keywords as evidence:
   If the abstract is absent or thin, treat keywords as supporting evidence.
   Climate-related keywords (e.g. "temperature", "seasonality", "climate change",
   "bioclimatic variables", "habitat suitability", "precipitation") satisfy the
   climate criterion even if not elaborated in the abstract.
   A keyword like "climate change" or "seasonal abundance" alone — with no
   contradicting abstract — is sufficient to satisfy the climate criterion.

RULE 2 — Climate as context counts:
   Label CLIMATE_TICK_STUDY if climate is invoked as an explanatory factor
   shaping tick outcomes — even without formal statistical testing of the
   climate-tick relationship.
   Label TICK_STUDY_NO_CLIMATE only if climate is entirely absent or appears
   in a single passing sentence unrelated to the study findings.

RULE 3 — Uncertainty:
   If the abstract is too short or ambiguous to judge, use label UNCERTAIN.
   Do NOT guess. Do NOT default to exclusion under uncertainty.

─────────────────────────────────────────────
OUTPUT FORMAT
─────────────────────────────────────────────

Return valid JSON only — no markdown, no preamble:
{
  "label": "CLIMATE_TICK_STUDY" | "TICK_STUDY_NO_CLIMATE" | "CLIMATE_NO_TICK" | "NOT_RELEVANT" | "UNCERTAIN",
  "verdict": "include" | "exclude" | "uncertain",
  "confidence": "high" | "medium" | "low",
  "notes": "one sentence explaining the key deciding factor"
}"""


# ── 3. INPUT BUILDER ─────────────────────────────────────────

def build_user_msg(title, abstract, keywords, screen_input):
    """
    Assemble the per-record prompt sent to the model.
    Keywords are treated as first-class evidence when the abstract is thin.
    """
    body = f"TITLE: {title}\n\n"

    if screen_input == "title+abstract" and abstract:
        body += f"ABSTRACT: {abstract}\n\n"
    else:
        body += (
            "ABSTRACT: [not available — screen on title and keywords only; "
            "if insufficient, mark uncertain]\n\n"
        )

    if keywords:
        body += (
            f"KEYWORDS: {keywords}\n\n"
            "NOTE: If the abstract is absent or thin, use the keywords above as "
            "supporting evidence for climate relevance. Climate-related keywords "
            "(e.g. 'temperature', 'seasonality', 'climate change', 'bioclimatic "
            "variables', 'habitat suitability', 'precipitation') satisfy the climate "
            "criterion even if not elaborated in the abstract.\n"
        )

    return body


# ── 4. JSON PARSER ───────────────────────────────────────────

def parse_json(txt):
    """Extract and parse the first JSON object from a model response."""
    if not txt:
        return None
    m = re.search(r'\{.*\}', txt, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


# ── 5. MODEL CALLERS ─────────────────────────────────────────

def call_claude(sys_prompt, user_msg):
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=300,
        system=sys_prompt,
        messages=[{"role": "user", "content": user_msg}]
    )
    return parse_json("".join(b.text for b in response.content if b.type == "text"))


def call_gpt(sys_prompt, user_msg):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_KEY)
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user",   "content": user_msg}
    ]
    # Try newer parameter name first, fall back for older SDK versions
    try:
        r = client.chat.completions.create(
            model=GPT_MODEL, max_completion_tokens=300, messages=messages
        )
    except TypeError:
        r = client.chat.completions.create(
            model=GPT_MODEL, temperature=0, max_tokens=300, messages=messages
        )
    return parse_json(r.choices[0].message.content)


def call_gemini(sys_prompt, user_msg):
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_KEY)
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=user_msg,
        config=types.GenerateContentConfig(
            system_instruction=sys_prompt,
            temperature=0,
            max_output_tokens=300
        )
    )
    return parse_json(response.text)


CALLERS = {"claude": call_claude, "gpt": call_gpt, "gemini": call_gemini}


def call_with_retry(fn, sys_prompt, user_msg, tries=3):
    """Call a model function with exponential-backoff retry."""
    for attempt in range(tries):
        try:
            result = fn(sys_prompt, user_msg)
            if result:
                return result
        except Exception as e:
            wait = 2 ** attempt
            print(f"    [retry {attempt+1}/{tries}] {type(e).__name__}: {e} — waiting {wait}s")
            time.sleep(wait)
    return None


# ── 6. SELF-TEST ─────────────────────────────────────────────

fn = CALLERS[MODEL_TO_RUN]
print(f"Self-test [{MODEL_TO_RUN}] ...", end=" ")

_test = call_with_retry(
    fn, CRITERIA,
    build_user_msg(
        title        = "Climate change alters Ixodes ricinus tick distribution",
        abstract     = "We modelled how rising temperature affects tick abundance and range expansion.",
        keywords     = "ticks; climate change; temperature; range expansion; habitat suitability",
        screen_input = "title+abstract"
    ),
    tries=2
)

if not _test or not _test.get("verdict"):
    raise SystemExit(
        f"\n[FAIL] {MODEL_TO_RUN} self-test failed — check API key or model name.\n"
        f"For Gemini confirm the model name at: https://ai.google.dev/gemini-api/docs/models\n"
        f"Nothing has been screened."
    )

print(
    f"OK  verdict={_test['verdict']} | "
    f"label={_test.get('label', '')} | "
    f"confidence={_test.get('confidence', '')}\n"
)


# ── 7. COLUMN NAMES FOR THIS MODEL ───────────────────────────

vcol = f"{MODEL_TO_RUN}_verdict"   # include / exclude / uncertain / error
lcol = f"{MODEL_TO_RUN}_label"     # detailed classification label
ccol = f"{MODEL_TO_RUN}_conf"      # high / medium / low
ncol = f"{MODEL_TO_RUN}_notes"     # one-sentence justification


# ── 8. LOAD DATAFRAME ────────────────────────────────────────

if os.path.exists(RESULTS):
    df = pd.read_excel(RESULTS, sheet_name="screening", dtype=str).fillna("")
    print(f"Loaded shared results file: {len(df)} records")
else:
    df = pd.read_excel(IN_PATH, sheet_name="canonical", dtype=str).fillna("")
    print(f"Started from screen-ready input: {len(df)} records")

if MAX_RECORDS:
    df = df.head(MAX_RECORDS)
    print(f"  (capped at {MAX_RECORDS} for testing)")

# Ensure all four output columns exist
for col in [vcol, lcol, ccol, ncol]:
    if col not in df.columns:
        df[col] = ""

# Informational: how many records have no abstract?
title_only_n = (df.get("screen_input", pd.Series(dtype=str)) == "title_only").sum()
print(f"  Title-only records (no abstract): {title_only_n} ({title_only_n/len(df)*100:.1f}%)")


# ── 9. RESUME FROM CHECKPOINT ────────────────────────────────
# Only restore rows that carry a real verdict to avoid blanking good
# data with empty strings from a partially-written checkpoint.

if os.path.exists(CKPT):
    prev  = pd.read_parquet(CKPT).set_index("record_id")
    valid = prev[vcol].isin(["include", "exclude", "uncertain", "error"])
    df    = df.set_index("record_id")
    df.update(prev.loc[valid])
    df    = df.reset_index()
    print(f"  Checkpoint restored: {valid.sum()} rows")

# Records still needing a verdict for this model
todo = df[~df[vcol].isin(["include", "exclude", "uncertain"])]
print(f"\n{MODEL_TO_RUN}: {len(todo)} to screen | {len(df)-len(todo)} already done\n")


# ── 10. MAIN SCREENING LOOP ──────────────────────────────────

t0 = time.time()

for n, (idx, row) in enumerate(todo.iterrows()):

    user_msg = build_user_msg(
        title        = row.get("title", ""),
        abstract     = row.get("abstract", ""),
        keywords     = row.get("keywords", ""),
        screen_input = row.get("screen_input", "title+abstract")
    )

    result = call_with_retry(fn, CRITERIA, user_msg)

    if result:
        df.at[idx, vcol] = result.get("verdict",    "")
        df.at[idx, lcol] = result.get("label",      "")
        df.at[idx, ccol] = result.get("confidence", "")
        df.at[idx, ncol] = result.get("notes",      "")
    else:
        df.at[idx, vcol] = "error"
        print(f"  [error] record index {idx} — all retries failed")

    # Rate limiting — prevents 429 errors on large runs
    time.sleep(RATE_LIMIT_SLEEP)

    # Checkpoint + progress log
    if n > 0 and n % CHECKPOINT_EVERY == 0:
        df.to_parquet(CKPT, index=False)
        elapsed = time.time() - t0
        rate    = (n + 1) / elapsed
        eta_s   = (len(todo) - n - 1) / rate if rate > 0 else 0
        dist    = df.loc[df[vcol].ne(""), vcol].value_counts().to_dict()
        print(
            f"  {n+1:>5}/{len(todo)} ({(n+1)/len(todo)*100:5.1f}%) | "
            f"{rate*60:5.1f} rec/min | "
            f"ETA {int(eta_s//3600)}h{int((eta_s%3600)//60):02d}m | "
            f"{dist}"
        )

# Final checkpoint
df.to_parquet(CKPT, index=False)


# ── 11. WRITE SHARED RESULTS FILE ────────────────────────────

def safe_xlsx(frame, path, sheet):
    """
    Write screening results to Excel.
    - Keeps only essential columns to avoid the 32k-char cell truncation
      that occurs with wide enrichment columns (affiliations, full-text…).
    - Strips illegal control characters that corrupt the xlsx file.
    """
    ILLEGAL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

    # Core metadata + all model output columns present in the frame
    keep = [
        "record_id", "title", "abstract", "keywords",
        "year", "journal", "doi", "screen_input",
        *[c for c in frame.columns if c.startswith(("gpt_", "claude_", "gemini_"))]
    ]
    out = frame[[c for c in keep if c in frame.columns]].copy()

    for col in out.columns:
        out[col] = out[col].map(lambda v: ILLEGAL.sub(" ", str(v))[:32000])

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        out.to_excel(writer, sheet_name=sheet, index=False)

    print(f"  Saved: {path}  ({len(out)} rows, {len(out.columns)} columns)")


safe_xlsx(df, RESULTS, "screening")


# ── 12. SUMMARY ──────────────────────────────────────────────

print(f"\n{'='*55}")
print(f"  {MODEL_TO_RUN.upper()} PASS COMPLETE")
print(f"{'='*55}")
print(f"  Verdicts : {df[vcol].value_counts().to_dict()}")
print(f"  Labels   : {df[lcol].value_counts().to_dict()}")
print(f"  Errors   : {(df[vcol] == 'error').sum()}")
print(f"  Output   : {RESULTS}")
print(f"{'='*55}")
print("\nNext step: run the other models (change MODEL_TO_RUN),")
print("then run 14b_consensus.py to combine all three verdicts.")